<a href="https://colab.research.google.com/github/glshrrkhan/Plant-Disease-Detection/blob/main/Notebook/Plant_Disease_Detection_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 Plant Disease Detection — Inference Notebook


This notebook loads the three trained models from Google Drive and launches
an interactive web application for real-time tomato leaf disease diagnosis.

**Notebook structure:**
- **Cell 1** — Mount Google Drive, import libraries, load all three trained models
- **Cell 2** — Grad-CAM utility functions for heatmap generation
- **Cell 3** — Gradio web interface for interactive disease detection

**What the app provides:**
- Drag-and-drop tomato leaf image upload
- Simultaneous prediction from Custom CNN, ResNet50, and MobileNetV2
- Confidence scores and top 3 predictions per model
- Grad-CAM heatmaps showing which leaf regions influenced each prediction

**Prerequisites:** Training notebook must have been run and all three models
saved to Google Drive at `MyDrive/PlantDiseaseDetectionUsingCNNs/models/`

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from google.colab import drive, files

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/PlantDiseaseDetectionUsingCNNs'
MODELS_DIR  = f'{PROJECT_DIR}/models'

CLASS_NAMES = sorted([
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
])
SHORT_NAMES = [c.replace('Tomato___', '').replace('_', ' ') for c in CLASS_NAMES]

print("Loading models from Drive...")
custom_cnn      = load_model(f'{MODELS_DIR}/custom_cnn_best.keras')
resnet_model    = load_model(f'{MODELS_DIR}/resnet50_best.keras')
mobilenet_model = load_model(f'{MODELS_DIR}/mobilenet_best.keras')
print("✅ All three models loaded successfully")

model_configs = {
    'Custom CNN':  (custom_cnn,      lambda x: x),
    'ResNet50':    (resnet_model,    lambda x: resnet_preprocess(x * 255)),
    'MobileNetV2': (mobilenet_model, lambda x: mobilenet_preprocess(x * 255))
}

grad_cam_layers = {
    'Custom CNN':  ('last_conv',          None),
    'ResNet50':    ('conv5_block3_out',   'resnet50'),
    'MobileNetV2': ('out_relu',           'mobilenetv2_1.00_224')
}

Mounted at /content/drive
Loading models from Drive...
✅ All three models loaded successfully


## Step 1 — Grad-CAM Utility Functions

Helper functions used by the inference pipeline to generate Grad-CAM
heatmaps and overlay them on the original image.

In [ ]:
def get_gradcam(model, img_array, conv_layer_name, nested_name=None):
    if nested_name:
        base        = model.get_layer(nested_name)
        feat_model  = Model(inputs=base.input, outputs=base.get_layer(conv_layer_name).output)
    else:
        feat_model  = Model(inputs=model.input, outputs=model.get_layer(conv_layer_name).output)

    with tf.GradientTape() as tape:
        features = feat_model(img_array)
        tape.watch(features)
        preds       = model(img_array)
        pred_index  = tf.argmax(preds[0])
        class_score = preds[:, pred_index]

    grads       = tape.gradient(class_score, features)
    if grads is None:
        heatmap = tf.reduce_mean(features[0], axis=-1)
    else:
        pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = features[0] @ pooled[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    colored         = plt.get_cmap('jet')(np.uint8(255 * heatmap))[:, :, :3]
    colored_resized = tf.image.resize(colored, (224, 224)).numpy()
    overlaid        = colored_resized * alpha + img
    return overlaid / overlaid.max()

print("✅ Grad-CAM functions ready")

✅ Grad-CAM functions ready


## Step 2 — Interactive Web Interface (Gradio)

This cell launches a professionally designed web application for tomato leaf
disease diagnosis. The interface is optimised for single-screenshot capture
on a laptop screen, displaying all three model predictions, confidence scores,
and Grad-CAM heatmaps in a compact, visually structured layout.

**Interface features:**
- Drag-and-drop image upload
- Simultaneous prediction from all three CNN models
- Confidence score with visual progress indicators
- Grad-CAM heatmaps showing model attention regions
- Clean dashboard layout designed for dissertation demonstration

In [ ]:
!pip install gradio -q

import gradio as gr
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

def predict_disease(input_image):
    if input_image is None:
        return None, None, None, "Please upload an image."

    img       = input_image.resize((224, 224))
    img_array = np.array(img) / 255.0

    results_text   = ""
    gradcam_images = []

    for model_name, (model, preprocess_fn) in model_configs.items():
        inp        = tf.cast(preprocess_fn(img_array.copy())[np.newaxis, ...], tf.float32)
        preds      = model(inp, training=False).numpy()[0]
        top_idx    = np.argmax(preds)
        confidence = preds[top_idx] * 100
        top3       = np.argsort(preds)[::-1][:3]

        bar = "█" * int(confidence / 5) + "░" * (20 - int(confidence / 5))
        results_text += f"{'─'*45}\n"
        results_text += f"  {model_name}\n"
        results_text += f"{'─'*45}\n"
        results_text += f"  🌿 Disease   : {SHORT_NAMES[top_idx]}\n"
        results_text += f"  📊 Confidence: {confidence:.1f}%  [{bar}]\n"
        results_text += f"  Top 3:\n"
        for rank, idx in enumerate(top3, 1):
            results_text += f"    {rank}. {SHORT_NAMES[idx]:<36} {preds[idx]*100:.1f}%\n"
        results_text += "\n"

        conv_layer, nested = grad_cam_layers[model_name]
        heatmap  = get_gradcam(model, inp, conv_layer, nested)
        overlaid = overlay_gradcam(img_array.copy(), heatmap)
        overlaid = (overlaid * 255).astype(np.uint8)
        gradcam_images.append(Image.fromarray(overlaid))

    return gradcam_images[0], gradcam_images[1], gradcam_images[2], results_text

css = """
* { box-sizing: border-box; }
body, .gradio-container {
    background: #0f1117 !important;
    font-family: 'Segoe UI', sans-serif !important;
}
.gradio-container {
    max-width: 1180px !important;
    margin: 0 auto !important;
    padding: 8px !important;
}
#header {
    background: linear-gradient(135deg, #1a6b3c, #2d9e5f);
    border-radius: 10px;
    padding: 12px 20px;
    margin-bottom: 10px;
    text-align: center;
    border: 1px solid #2d9e5f;
}
#header h1 {
    color: #ffffff !important;
    font-size: 1.4em !important;
    margin: 0 !important;
    font-weight: 700 !important;
}
#header p {
    color: #a8f0c6 !important;
    margin: 3px 0 0 0 !important;
    font-size: 0.8em !important;
}
label {
    color: #a8f0c6 !important;
    font-weight: 600 !important;
    font-size: 0.78em !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}
.gr-button-primary {
    background: linear-gradient(135deg, #1a6b3c, #2d9e5f) !important;
    border: none !important;
    color: white !important;
    font-weight: 700 !important;
    font-size: 0.95em !important;
    border-radius: 8px !important;
    padding: 10px !important;
    width: 100% !important;
    margin-top: 6px !important;
}
.gr-button-primary:hover {
    background: linear-gradient(135deg, #2d9e5f, #3dbf72) !important;
}
textarea {
    background: #0f1117 !important;
    color: #e0ffe0 !important;
    border: 1px solid #2a2d3e !important;
    border-radius: 8px !important;
    font-family: 'Courier New', monospace !important;
    font-size: 0.74em !important;
}
.image-container img {
    border-radius: 8px !important;
    border: 1px solid #2a2d3e !important;
}
footer { display: none !important; }
"""

with gr.Blocks(css=css, title="🌿 Plant Disease Detection") as app:

    gr.HTML("""
    <div id="header">
        <h1>🌿 Tomato Plant Disease Detection</h1>
        <p>MSc Dissertation &nbsp;|&nbsp; University of East London &nbsp;|&nbsp; CN7000 &nbsp;|&nbsp; Three-Model CNN Comparison</p>
    </div>
    """)

    with gr.Row():
        # Left column — upload + button + results
        with gr.Column(scale=1, min_width=300):
            input_img = gr.Image(type='pil', label="Upload Tomato Leaf Image", height=220)
            btn = gr.Button("🔍  Analyse Disease", variant="primary")
            results_box = gr.Textbox(
                label="Prediction Results — All Three Models",
                lines=16, interactive=False
            )

        # Right column — grad-cam + summary
        with gr.Column(scale=2):
            gr.HTML('<p style="color:#a8f0c6;font-weight:600;font-size:0.78em;text-transform:uppercase;letter-spacing:0.5px;margin:0 0 6px 0;">Grad-CAM Heatmaps — Model Attention Visualization</p>')
            with gr.Row():
                out_cnn       = gr.Image(label="Custom CNN",  height=200)
                out_resnet    = gr.Image(label="ResNet50",    height=200)
                out_mobilenet = gr.Image(label="MobileNetV2", height=200)

            gr.HTML("""
            <div style="background:#0f1117;border:1px solid #2a2d3e;border-radius:8px;padding:10px;margin-top:8px;">
                <p style="color:#a8f0c6;font-size:0.72em;margin:0 0 8px 0;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;">Model Performance Summary</p>
                <div style="display:flex;gap:8px;">
                    <div style="flex:1;background:#1a1d27;border-radius:6px;padding:8px;border:1px solid #2a2d3e;text-align:center;">
                        <p style="color:#64b5f6;font-size:0.7em;margin:0;font-weight:600;">CUSTOM CNN</p>
                        <p style="color:#ffffff;font-size:1.2em;margin:2px 0;font-weight:700;">84.87%</p>
                        <p style="color:#666;font-size:0.65em;margin:0;">Baseline &nbsp;|&nbsp; 220.6 MB</p>
                    </div>
                    <div style="flex:1;background:#1a1d27;border-radius:6px;padding:8px;border:1px solid #2d9e5f;text-align:center;">
                        <p style="color:#81c784;font-size:0.7em;margin:0;font-weight:600;">RESNET50 ★ BEST</p>
                        <p style="color:#ffffff;font-size:1.2em;margin:2px 0;font-weight:700;">96.80%</p>
                        <p style="color:#666;font-size:0.65em;margin:0;">Transfer Learning &nbsp;|&nbsp; 206.9 MB</p>
                    </div>
                    <div style="flex:1;background:#1a1d27;border-radius:6px;padding:8px;border:1px solid #2a2d3e;text-align:center;">
                        <p style="color:#ffb74d;font-size:0.7em;margin:0;font-weight:600;">MOBILENETV2</p>
                        <p style="color:#ffffff;font-size:1.2em;margin:2px 0;font-weight:700;">89.07%</p>
                        <p style="color:#666;font-size:0.65em;margin:0;">Lightweight &nbsp;|&nbsp; 24.6 MB</p>
                    </div>
                </div>
            </div>
            """)

            gr.HTML("""
            <div style="background:#0f1117;border:1px solid #2a2d3e;border-radius:8px;padding:8px;margin-top:6px;">
                <p style="color:#666;font-size:0.68em;margin:0;text-align:center;">
                    Dataset: PlantVillage — 10,000 images | 10 Tomato Disease Classes | 70/15/15 Split
                    &nbsp;|&nbsp; Sahil &nbsp;|&nbsp; UEL 2026
                </p>
            </div>
            """)

    btn.click(
        fn=predict_disease,
        inputs=input_img,
        outputs=[out_cnn, out_resnet, out_mobilenet, results_box]
    )

app.launch(share=True)

/tmp/ipykernel_3112/2817547012.py:113: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="🌿 Plant Disease Detection") as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36bc2a6db68f6eac1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
